In [ ]:
import pandas as pd
from pandas import DataFrame
import pymysql
import rp  
import ratespricer as rp
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os


# Parse a contract code, for example T2606 -> ('T', 26, 6)
def parse_contract_code(contract_code):
    prefix = ''.join(ch for ch in contract_code if ch.isalpha())
    yy = int(contract_code[len(prefix):len(prefix) + 2])
    mm = int(contract_code[-2:])
    return prefix, yy, mm


# Given a quarterly contract, return the next quarterly contract
def next_quarter_contract(contract_code):
    prefix, yy, mm = parse_contract_code(contract_code)
    quarter_months = [3, 6, 9, 12]

    if mm not in quarter_months:
        raise ValueError(f'Contract month must be one of {quarter_months}: {contract_code}')

    current_idx = quarter_months.index(mm)
    next_idx = (current_idx + 1) % len(quarter_months)
    next_yy = yy + 1 if next_idx == 0 else yy
    next_mm = quarter_months[next_idx]
    return f"{prefix}{next_yy:02d}{next_mm:02d}"




# Infer the nearby contract from the current date
def get_auto_near_contract(prefix, current_dt=None):
    if current_dt is None:
        current_dt = datetime.now()

    yy = current_dt.year % 100
    month_day = (current_dt.month, current_dt.day)

    if (3, 1) <= month_day < (6, 1):
        contract_year = yy
        contract_month = 6
    elif (6, 1) <= month_day < (9, 1):
        contract_year = yy
        contract_month = 9
    elif (9, 1) <= month_day < (12, 1):
        contract_year = yy
        contract_month = 12
    else:
        # For dates before Mar 1 or on/after Dec 1, roll to next year's Mar contract.
        contract_year = yy + 1 if month_day >= (12, 1) else yy
        contract_month = 3

    return f"{prefix}{contract_year:02d}{contract_month:02d}"

# Given a quarterly contract, return the next quarterly contract??? fut_list
def generate_fut_list_from_near(start_date, start_near, current_near):
    start_prefix, start_yy, start_mm = parse_contract_code(start_near)
    current_prefix, current_yy, current_mm = parse_contract_code(current_near)
    quarter_months = [3, 6, 9, 12]

    if start_prefix != current_prefix:
        raise ValueError('start_near and current_near must have the same prefix')
    if start_mm not in quarter_months or current_mm not in quarter_months:
        raise ValueError('Contract month must be one of 03/06/09/12')
    if (current_yy, current_mm) < (start_yy, start_mm):
        raise ValueError('current_near cannot be earlier than start_near')

    fut_list = []
    roll_date = datetime.strptime(start_date, '%Y-%m-%d')
    near_contract = start_near

    while True:
        far_contract = next_quarter_contract(near_contract)
        fut_list.append((roll_date.strftime('%Y-%m-%d'), near_contract, far_contract))

        if near_contract == current_near:
            break

        near_contract = far_contract
        next_month = roll_date.month + 3
        next_year = roll_date.year
        if next_month > 12:
            next_month -= 12
            next_year += 1
        roll_date = datetime(next_year, next_month, 1)

    return fut_list


def process_futures_data(fut_list):
    """
    TODO：最后交易日
    获取交割月日前80到前1个交易日的持仓数据
    返回每个合约品种（TL/T/TF/TS)所有合约(TL2403,TL2406,...,最远月合约）：
        posit      近月持仓量
        posit_perc 近月移仓进度 = 远月 / (近月+远月)
    """

    posit = DataFrame()        # 存近月持仓量
    posit_perc = DataFrame()   # 存近月移仓进度

    # 建立数据库连接
    cnn = pymysql.connect(host='192.168.119.53',
                          user='fangyd',
                          passwd='fyd2025!',
                          db='bond2',
                          charset='utf8')
    cur = cnn.cursor()

    # 遍历每组换月合约 (交割日, 近月, 远月)
    for ed, fut_code1, fut_code2 in fut_list:

        sql = '''
            SELECT DateTime, position 
            FROM futures_records_1d_origin 
            WHERE FutCode = '%s' AND DateTime BETWEEN '%s' and '%s'
            '''

        # 计算时间区间：交割日-80（个交易日） 到 交割日-1
        start = rp.d_get_bus_day(ed, -80).strftime('%Y-%m-%d')
        end   = rp.d_get_bus_day(ed, -1).strftime('%Y-%m-%d')

        # 查询近月持仓
        cur.execute(sql % (fut_code1, start, end))
        rst1 = cur.fetchall()

        # 查询远月持仓
        cur.execute(sql % (fut_code2, start, end))
        rst2 = cur.fetchall()

        # 转为 DataFrame
        fut1 = DataFrame.from_records(rst1, columns=['datetimes', 'position'])
        fut2 = DataFrame.from_records(rst2, columns=['datetimes', 'position'])

        # 保存近月持仓
        posit[fut_code1] = fut1['position']

        # 计算移仓进度
        posit_perc[fut_code1] = fut2['position'] / (fut1['position'] + fut2['position'])

    # 索引改为 -80 ~ -1
    posit.index = posit.index - 80
    posit_perc.index = posit_perc.index - 80

    cur.close()
    cnn.close()

    return posit, posit_perc




# Plot open-interest changes and highlight the nearby contract from nearby_contract_map
def plot_combined_oi(posit_dict, contract_types, nearby_contract_map):
    """
    将多个合约持仓量绘制在 2×2 网格图中
    posit_dict: {'TL': df, 'T': df, ...}
    contract_types: ['TL','T','TF','TS']
    """

    # 解决中文与负号显示问题
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    # 创建 2×2 子图
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    axs = axs.flatten()  # 展平成一维数组便于循环

    # 遍历每种合约类型
    for i, contract_type in enumerate(contract_types):
        if i >= len(axs):  # 防止超出子图数量
            break

        ax = axs[i]
        posit = posit_dict[contract_type]  # 当前合约数据
        contracts = list(posit.columns)    # 合约列表
        # TODO：需修改
        Nearby_contract = nearby_contract_map.get(contract_type)
        print(f'近月合约：{Nearby_contract}')

        # 绘制每个合约曲线
        for contract in contracts:
            is_Nearby = (contract == Nearby_contract)

            # 设置样式（最后一个合约加粗红色）
            ax.plot(
                posit.index,
                posit[contract],
                label=contract,
                color='red' if is_Nearby else None,
                linewidth=2 if is_Nearby else 1,
                marker='o',
                markersize=5 if is_Nearby else 3
            )

            # 若为近月合约，标注最后一个有效值
            if is_Nearby:
                valid = posit[contract].dropna()
                if not valid.empty:
                    ax.text(
                        valid.index[-1],
                        valid.iloc[-1],
                        f'{valid.iloc[-1]:.0f}',
                        fontsize=11,
                        fontweight='bold',
                        ha='left',
                        va='bottom'
                    )

        # 子图格式设置
        ax.set_title(f'{contract_type}合约持仓量变化', fontsize=13)
        ax.set_xlabel('距交割月日天数')
        ax.set_ylabel('持仓量')
        ax.legend(loc='lower left', fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.6)

        if not posit.empty:
            ax.set_xlim(posit.index.min(), posit.index.max())

    plt.tight_layout()

    # === 保存图片 ===
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    daily_folder = f'????_{today}'
    file_path_folder = os.path.join(result_folder, daily_folder)
    os.makedirs(file_path_folder, exist_ok=True)

    filename = f'?????????_{today}.png'
    file_path = os.path.join(file_path_folder, filename)
    print(f'图片已保存: {file_path}')

    plt.show()


# 绘制移仓进度对比图（含持仓量变动）
# Plot the 2x2 overview: roll progress on the left axis and nearby OI changes on the right axis
def plot_roll_overview(data_dict, nearby_contract_map):
    """
    # (open interest, roll progress) data used by the overview and contrast charts
    data_dict = {
        'TL': (posit, posit_perc),
        'T':  (posit, posit_perc),
        ...
    }
    """
    
    # 设置中文字体和负号正常显示
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    # 创建2×2子图
    fig, axes = plt.subplots(2, 2, figsize=(22, 16))
    axes = axes.flatten()  # 展平便于循环

    today = datetime.now().strftime('%Y%m%d')  # 获取今日日期

    # 遍历每个品种
    for i, (product, (posit, posit_perc)) in enumerate(data_dict.items()):

        ax = axes[i]  # 当前子图

        Nearby_contract = nearby_contract_map.get(product)
        print(f'近月合约：{Nearby_contract}')

        # ===== 左轴：移仓进度曲线 =====
        for col in posit_perc.columns:

            is_Nearby = (col == Nearby_contract)

            # ★修改：所有合约都改为虚线
            ax.plot(
                posit_perc.index,
                posit_perc[col],
                linestyle='--',              # ★修改
                linewidth=3 if is_Nearby else 1.5,
                color='red' if is_Nearby else None,
                label=col,
                zorder=2                    # ★新增：控制图层
            )

            # 近月标注最后一个有效点
            if is_Nearby:
                valid = posit_perc[col].dropna()
                if not valid.empty:
                    last_x = valid.index[-1]
                    last_y = valid.iloc[-1]

                    ax.scatter(last_x, last_y, s=80, color='darkred', zorder=5)  # ★修改zorder

                    # ★修改：置于柱子之上
                    ax.annotate(
                        f"({last_x:.0f}, {last_y:.3f})",
                        (last_x, last_y),
                        xytext=(10, 10),
                        textcoords='offset points',
                        fontsize=16,
                        fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3',
                                  facecolor='red',
                                  alpha=0.9),
                        arrowprops=dict(arrowstyle='->',
                                        color='darkred'),
                        zorder=6  # ★新增：最高图层
                    )

        # 左轴格式设置
        ax.set_title(f"{product}移仓进度", fontsize=14, fontweight='bold')
        ax.set_xlabel('距交割月日天数')
        ax.set_ylabel('移仓进度')
        ax.set_xlim(-80, 0)
        ax.set_ylim(0, 1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(fontsize=8, loc='upper left')

        # ===== 右轴：近月持仓变动 =====
        ax_right = ax.twinx()  # 创建右侧Y轴

        if Nearby_contract in posit.columns:

            oi = posit[Nearby_contract].diff().dropna()

            x_vals = oi.index
            y_vals = oi.values

            colors = ['red' if y > 0 else 'green' for y in y_vals]

            # ★修改：增加黑色实线边框
            bars = ax_right.bar(
                x_vals,
                y_vals,
                color=colors,
                edgecolor='black',      # ★新增
                linewidth=0.6,          # ★新增
                alpha=0.7,
                width=1.0,
                label=f"{Nearby_contract}持仓量变动",
                zorder=1               # ★新增：柱子在底层
            )

            ax_right.set_ylabel("持仓量变动")
            ax_right.tick_params(axis='y')

            if len(y_vals) > 0:
                max_val = max(abs(y_vals.max()), abs(y_vals.min()))
                ax_right.set_ylim(-max_val * 1.2, max_val * 1.2)

            # 标注最后一个柱子
            if len(x_vals) > 0:
                last_x = x_vals[-1]
                last_y = y_vals[-1]

                bbox_color = 'lightcoral' if last_y > 0 else 'lightgreen'
                edge_color = 'darkred' if last_y > 0 else 'darkgreen'

                ax_right.annotate(
                    f'{last_y:.0f}',
                    xy=(last_x, last_y),
                    xytext=(0, 15),
                    textcoords='offset points',
                    ha='center',
                    fontsize=16,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor=bbox_color,
                              edgecolor=edge_color),
                    arrowprops=dict(arrowstyle='->',
                                    color=edge_color),
                    zorder=3   # ★新增：在柱子之上但低于移仓点
                )

            ax_right.legend(loc='upper right', fontsize=8)

    # 设置总标题
    plt.suptitle("国债期货移仓进度总览（含近月合约持仓量变动）",
                 fontsize=18,
                 fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)

    # === 保存图片 ===
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    daily_folder = f'????_{today}'
    file_path_folder = os.path.join(result_folder, daily_folder)
    os.makedirs(file_path_folder, exist_ok=True)

    filename = f'??????????????????_{today}.png'
    file_path = os.path.join(file_path_folder, filename)
    print(f'图片已保存: {file_path}')

    plt.show()


# Plot roll-progress contrast for the nearby contract of each product
def plot_roll_contrast(data_dict, nearby_contract_map):
    """
    画四个品种（TL/T/TF/TS）近月合约移仓进度对比图
    # (open interest, roll progress) data used by the overview and contrast charts
    data_dict = {
        'TL': (posit, posit_perc),
        'T':  (posit, posit_perc),
        'TF': (posit, posit_perc),
        'TS': (posit, posit_perc),
    }
    """

    # 设置中文字体和负号正常显示
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    fig, ax = plt.subplots(figsize=(16, 10))

    # 为四个品种指定固定颜色（与示意图风格一致）
    color_map = {
        'TL': 'blue',
        'T': 'red',
        'TF': 'green',
        'TS': 'orange'
    }

    for product, (posit, posit_perc) in data_dict.items():

        if posit_perc.empty:
            continue

        # 近月合约（与你原逻辑保持一致）
        Nearby_contract = nearby_contract_map.get(product)

        x = posit_perc.index
        y = posit_perc[Nearby_contract]

        ax.plot(
            x,
            y,
            linestyle='--',
            linewidth=3,
            color=color_map.get(product, None),
            label=f"{Nearby_contract}",
            zorder=2
        )

        # 标注最后一个有效点
        valid = y.dropna()
        if not valid.empty:
            last_x = valid.index[-1]
            last_y = valid.iloc[-1]

            ax.scatter(
                last_x,
                last_y,
                s=120,
                color=color_map.get(product, None),
                zorder=5
            )

            ax.annotate(
                f"({last_x:.0f}, {last_y:.3f})",
                (last_x, last_y),
                xytext=(10, 10),
                textcoords='offset points',
                fontsize=14,
                fontweight='bold',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor=color_map.get(product, 'gray'),
                    alpha=0.8
                ),
                arrowprops=dict(
                    arrowstyle='->',
                    color=color_map.get(product, 'black')
                ),
                zorder=6
            )

    # 图形格式
    ax.set_title("各合约移仓进度对比", fontsize=18, fontweight='bold')
    ax.set_xlabel("距交割月日天数")
    ax.set_ylabel("移仓进度")
    ax.set_xlim(-80, 0)
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=12)

    plt.tight_layout()

    # 保存图片
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    daily_folder = f'????_{today}'
    file_path_folder = os.path.join(result_folder, daily_folder)
    os.makedirs(file_path_folder, exist_ok=True)

    filename = f'??????????_{today}.png'
    file_path = os.path.join(file_path_folder, filename)
    print(f'图片已保存: {file_path}')

    plt.show()

# Plot roll-progress contrast for the nearby contract of each product
def plot_roll_contrast(data_dict, nearby_contract_map):
    """
    四大品种近月合约移仓进度对比图
    Y轴自适应：
        - 最大值 > 0.9 时封顶 1.0
        - 否则按 0.1 为最小单位向上取整 + 0.1 缓冲
    """

    # ===== 基础图形设置 =====
    plt.rcParams['font.sans-serif'] = ['SimHei']      # 解决中文显示
    plt.rcParams['axes.unicode_minus'] = False        # 解决负号显示

    fig, ax = plt.subplots(figsize=(16, 10))          # 创建单图

    # ===== 颜色映射（固定品种颜色）=====
    color_map = {
        'TL': 'blue',
        'T': 'red',
        'TF': 'green',
        'TS': 'orange'
    }

    # ===== 标注偏移映射（避免重叠）=====
    offset_map = {
        'TL': (15, 20),
        'T':  (15, -30),
        'TF': (-80, 20),
        'TS': (-80, -30)
    }

    global_max_y = 0   # 记录四个品种中的最大移仓值

    # ===== 遍历每个品种 =====
    for product, (posit, posit_perc) in data_dict.items():

        if posit_perc.empty:      # 若数据为空则跳过
            continue

        Nearby_contract = nearby_contract_map.get(product)

        x = posit_perc.index                      # 横轴：距离交割日天数
        y = posit_perc[Nearby_contract]           # 纵轴：近月移仓进度

        # 更新全局最大值
        local_max = y.max()
        if pd.notna(local_max):
            global_max_y = max(global_max_y, local_max)

        # ===== 绘制移仓进度曲线 =====
        ax.plot(
            x,
            y,
            linestyle='--',                        # 虚线
            linewidth=3,                           # 加粗
            color=color_map.get(product, None),    # 固定颜色
            label=Nearby_contract,                 # 图例显示合约
            zorder=2                               # 图层控制
        )

        # ===== 标注最后一个有效点 =====
        valid = y.dropna()                         # 去除空值
        if not valid.empty:

            last_x = valid.index[-1]               # 最后一个X
            last_y = valid.iloc[-1]                # 最后一个Y

            # 绘制终点圆点
            ax.scatter(
                last_x,
                last_y,
                s=120,
                color=color_map.get(product, None),
                zorder=5
            )

            # 获取对应偏移方向
            dx, dy = offset_map.get(product, (10, 10))

            # 添加带箭头标注
            ax.annotate(
                f"({last_x:.0f}, {last_y:.3f})",
                (last_x, last_y),
                xytext=(dx, dy),
                textcoords='offset points',
                fontsize=14,
                fontweight='bold',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor=color_map.get(product, 'gray'),
                    alpha=0.85
                ),
                arrowprops=dict(
                    arrowstyle='->',
                    color=color_map.get(product, 'black')
                ),
                zorder=6
            )

    # ===== Y轴自适应逻辑 =====

    if global_max_y > 0.9:
        upper = 1.0     # 若接近完成移仓，则封顶1.0
    elif global_max_y == 0:
        upper = 0.1     # 若数据异常，则给最小刻度
    else:
        # 向上取整到0.1倍数
        upper = np.ceil(global_max_y / 0.1) * 0.1
        upper += 0.1    # 额外留出0.1缓冲空间

    # 设置坐标轴范围
    ax.set_ylim(0, upper)
    ax.set_xlim(-80, 0)

    # ===== 图形格式 =====
    ax.set_title("各合约移仓进度对比", fontsize=18, fontweight='bold')
    ax.set_xlabel("距离近月合约最后交易日天数")
    ax.set_ylabel("移仓进度")
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=12)

    plt.tight_layout()

    # ===== 保存图片 =====
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    daily_folder = f'????_{today}'
    file_path_folder = os.path.join(result_folder, daily_folder)
    os.makedirs(file_path_folder, exist_ok=True)

    filename = f'??????????_{today}.png'
    file_path = os.path.join(file_path_folder, filename)
    print(f'图片已保存: {file_path}')

    plt.show()

if __name__ == "__main__":
    # 自动生成当前近月合约
    fut_TL_near = get_auto_near_contract('TL')
    fut_T_near = get_auto_near_contract('T')
    fut_TF_near = get_auto_near_contract('TF')
    fut_TS_near = get_auto_near_contract('TS')

    # 根据近月合约生成移仓换月列表
    fut_TL_list = generate_fut_list_from_near('2024-03-01', 'TL2403', fut_TL_near)
    fut_T_list = generate_fut_list_from_near('2024-03-01', 'T2403', fut_T_near)
    fut_TF_list = generate_fut_list_from_near('2024-03-01', 'TF2403', fut_TF_near)
    fut_TS_list = generate_fut_list_from_near('2024-03-01', 'TS2403', fut_TS_near)

    # 汇总当前近月合约映射
    nearby_contract_map = {
        'TL': fut_TL_near,
        'T': fut_T_near,
        'TF': fut_TF_near,
        'TS': fut_TS_near,
    }

    # 查询各品种数据
    posit_TL, posit_perc_TL = process_futures_data(fut_TL_list)
    posit_T, posit_perc_T = process_futures_data(fut_T_list)
    posit_TF, posit_perc_TF = process_futures_data(fut_TF_list)
    posit_TS, posit_perc_TS = process_futures_data(fut_TS_list)

    # 汇总持仓数据字典
    posit_dict = {
        'TL': posit_TL,
        'T': posit_T,
        'TF': posit_TF,
        'TS': posit_TS,
    }

    # 汇总绘图输入数据字典
    data_dict = {
        'TL': (posit_TL, posit_perc_TL),
        'T': (posit_T, posit_perc_T),
        'TF': (posit_TF, posit_perc_TF),
        'TS': (posit_TS, posit_perc_TS),
    }

    # 绘制持仓变化图
    plot_combined_oi(posit_dict, ['TL', 'T', 'TF', 'TS'], nearby_contract_map)

    # 绘制移仓总览图
    plot_roll_overview(data_dict, nearby_contract_map)

    # 绘制移仓对比图
    plot_roll_contrast(data_dict, nearby_contract_map)
